使用一个聪明的模型提取答案，按照提取LLM的格式输出
只运行单一的json
限制回答问题的LLM的输出，限定了给定格式

In [ ]:
import json
import os
from PIL import Image
import re
import ast
import base64
import time
from openai import OpenAI
import io

In [12]:
def generate_prompt_q1(sample):
    question_comp = sample["question_comp"]
    items = sample["items"]
    caption_map = sample["caption"]
    
    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)
    
    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Question:
{question_comp}

Instructions:
Analyze the provided items to answer the question.
"""

    return prompt

In [13]:
def generate_prompt_q2(q1_resp, sample):
    items = sample["items"]
    caption_map = sample["caption"]
    question_gen = sample["question_gen"]
    question_comp = sample["question_comp"]

    # 输出格式规则
    format_rule = "[[...], [...], ...]"

    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)

    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Previous Question:
{question_comp}

Previous Answer:
{q1_resp}

Question:
{question_gen}

Instructions:
1. Analyze the provided items and the Q1 answer to answer the Q2 question.
2. Do NOT output reasoning, explanation, or intermediate steps.
3. Output ONLY the final answer.

Output format:
Answer: {format_rule}

Rules:
- Output exactly ONE line.
- The line must start with "Answer:".
- Do not output anything before or after the answer.
"""

    return prompt

In [14]:
# 转换图表格式 压缩图片-> 减少token
def encode_image_to_base64(image_path, max_size=768, quality=70):
    img = Image.open(image_path)

    # 等比例缩小
    img.thumbnail((max_size, max_size))
    
    if img.mode != "RGB":
        img = img.convert("RGB")

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
def call_llm_multiround(subject_path, sample):
    
    # 生成提示词
    prompt_q1 = generate_prompt_q1(sample)

    content_list_q1 = []
    path_map = sample["path"]
    

    for key in path_map:
        relative_path = path_map[key]
        full_path = os.path.join(subject_path, relative_path)

        if not os.path.exists(full_path):
            print(f"文件不存在: {full_path}")
            continue

        mime_type = "image/jpeg"
        image_base64 = encode_image_to_base64(full_path)

        content_list_q1.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_base64}"
            }
        })
        
    content_list_q2 = content_list_q1.copy()

    # 添加文本
    content_list_q1.append({
        "type": "text",
        "text": prompt_q1
    })
    

    
    # 调用模型
    
    client = OpenAI(
        api_key="",
        base_url=""
    )   

    attempt = 0
    base_sleep = 1
    max_sleep= 60
    MAX_RETRY = 50 # 避免死循环
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
        
        try:
            # Q1
            resp_q1 = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list_q1}],
                temperature=0
            )
            q1_resp = resp_q1.choices[0].message.content
            
            print(f"answer1 = {q1_resp}\n")
            
            # Q2
            prompt_q2 = generate_prompt_q2(q1_resp, sample)
            content_list_q2.append({
                "type": "text",
                "text": prompt_q2
            })
            
            resp_q2 = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list_q2}],
                temperature=0
            )
            q2_resp = resp_q2.choices[0].message.content

            return q2_resp
        
        except Exception as e:
            msg = str(e).lower()
            
            if ("401" in msg or "403" in msg or "400" in msg or "invalid" in msg or "not found" in msg):
                raise
            
            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1
    

In [ ]:
# 使用deepseek提取答案
def exact_answer_llm(output, answer_type):
    
    # 设计prompt
        # 输出规则 
    if answer_type == "numeric":
        output_format = "answer: {number}. Return exactly one number after 'answer:'. Do not include units, symbols, commas, explanations, or extra text."

    elif answer_type == "numeric_list":
        output_format = "answer: [a, b, c]. Return a list of numbers inside square brackets. No text inside brackets. No explanations."

    elif answer_type == "choice":
        output_format = "answer: {A capital letter}. Return exactly one capital letter after 'answer:'. No punctuation. No brackets. No extra text."

    elif answer_type == "choice_list":
        output_format = "answer: [A, B, C]. Return a list of capital letters inside square brackets. No explanations."

    elif answer_type == "table_list":
        output_format = "answer: [[row1_col1, row1_col2, ...], [row2_col1, row2_col2, ...], ...]. Return a nested list representing the table. Only structured list format. "

    elif answer_type == "code":
        output_format = "answer: ```python {code block}```. Return only executable code inside the code block. No explanation outside the code block."

    else:
        output_format = "answer: {text}. Return only the final short text answer after 'answer:'. The answer MUST end with a period '.'. Do not include anything after the period. No explanations or additional commentary."
        
    
    prompt = f"""
You are an answer extraction assistant. Your task is to extract the final answer from a model’s response according to the specified answer type. 
Answer type: {answer_type}
Model response:
{output}
You must extract only the final answer and strictly follow this output format:
{output_format}
Do not include explanations or any additional text.
Return exactly one answer in the required format.
    """
        
    # 调用llm 多次调用 直到成功返回 deepseek
    client = OpenAI(
        api_key="", 
        base_url=""
    )

    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50 # 避免死循环
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
    
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": prompt}
            ],
                max_tokens=1024,
                temperature=0,
                stream=False
            )

            output_llm = response.choices[0].message.content
            
            if not output_llm:
                raise ValueError("empty response")
                
            return output_llm
 
        except Exception as e:
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1

In [18]:
# 测评代码 —— table_list

# 标准化行名和列名
def normalize_label(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 计算两个字符串的编辑距离
def _edit_distance(a, b):
    """
    Levenshtein edit distance (insert/delete/substitute).
    """
    a = normalize_label(a)
    b = normalize_label(b)
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n

    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(
                prev[j] + 1,        # delete
                cur[j - 1] + 1,     # insert
                prev[j - 1] + cost  # substitute
            )
        prev = cur
    return prev[m]

# 计算字符串相似度
def esm_sim(a, b):
    """
    ESM similarity:
    sim = 1 - ED(a,b) / max(len(a), len(b))
    """
    a_n = normalize_label(a)
    b_n = normalize_label(b)
    if len(a_n) == 0 and len(b_n) == 0:
        return 1.0
    if len(a_n) == 0 or len(b_n) == 0:
        return 0.0
    dist = _edit_distance(a_n, b_n)
    return 1.0 - dist / max(len(a_n), len(b_n))


# 计算数值相似度
def num_sim(p, g, eps=1e-12):
    """
    DePlot-style numeric distance and similarity:
      dist = min(1, |p-g| / max(|g|, eps))
      sim  = 1 - dist
    """
    try:
        if p is None or g is None:
            return 0.0
        p = float(p)
        g = float(g)
        denom = max(abs(g), eps)
        dist = abs(p - g) / denom
        dist = min(1.0, dist)
        return 1.0 - dist
    except Exception:
        return 0.0
    
    
# 将每个单元格整理成三元组 cells: list of (row_label, col_label, value)
def table_to_cells(table):

    # 1) 如果 table 是字符串（LLM输出常见），先转成 list
    if isinstance(table, str):
        try:
            table = ast.literal_eval(table.strip())
        except Exception:
            return []

    # 2) 基本结构检查
    if not isinstance(table, list) or len(table) < 2:
        return []

    if not all(isinstance(r, list) and len(r) >= 1 for r in table):
        return []

    # 3) 取列名（第一行除第一个）
    cols = table[0][1:]
    cells = []

    # 4) 遍历数据行
    for r in range(1, len(table)):
        row = table[r]
        if len(row) == 0:
            continue

        row_label = row[0]
        vals = row[1:]

        for j, col_label in enumerate(cols):
            val = vals[j] if j < len(vals) else None
            cells.append((row_label, col_label, val))

    return cells
    
# 匹配算法
# 简单贪心匹配
def greedy_max(weight):
    """
    最大化 sum(weight[i][j]) 的贪心一对一匹配
    返回 assignment: 长度 n，assignment[i] = j 或 -1
    """
    n = len(weight)
    m = len(weight[0]) if n else 0

    assignment = [-1] * n
    used_cols = set()

    # 收集所有可能匹配 (sim, i, j)
    pairs = []
    for i in range(n):
        for j in range(m):
            if weight[i][j] > 0:
                pairs.append((weight[i][j], i, j))

    # 按相似度降序排序
    pairs.sort(reverse=True)

    # 贪心匹配
    for sim, i, j in pairs:
        if assignment[i] == -1 and j not in used_cols:
            assignment[i] = j
            used_cols.add(j)

    return assignment


# 数字正则（你原来代码里用到的 _NUM）
_NUM = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def parse_table_list(output):
    try:
        s = str(output).strip()
        
        output_llm = exact_answer_llm(s, "table_list")

        # 1) 先从 LLM 输出中提取 answer: 后面的“第一个二维 list”
        #    - 要求是嵌套列表：[[...], [...], ...]
        #    - 用非贪婪匹配，尽量拿到最靠前的那个
        m = re.search(
            r"(?is)^\s*answer:\s*(\[\s*\[.*?\]\s*(?:,\s*\[.*?\]\s*)*\])",
            s
        )
        if m:
            table_str = m.group(1)
        else:
            m2 = re.search(r"(?is)(\[\s*\[.*?\]\s*(?:,\s*\[.*?\]\s*)*\])", s)
            if not m2:
                return None
            
            table_str = m2.group(1)

        # 2) token 级修复：把非数字的 label 自动加引号；数字/数字字符串保持数值
        pattern = r'(?<=\[|,)\s*([^\[\],]+?)\s*(?=,|\])'

        def fix_token(m):
            tok = m.group(1).strip()

            # 已带引号
            if (len(tok) >= 2) and (tok[0] == tok[-1]) and tok[0] in ("'", '"'):
                inner = tok[1:-1].strip()
                if _NUM.fullmatch(inner):
                    return inner
                inner_esc = inner.replace('"', '\\"')
                return f'"{inner_esc}"'

            # 不带引号：数字
            if _NUM.fullmatch(tok):
                return tok

            # 不带引号且非数字：视为字符串 label
            tok_esc = tok.replace('"', '\\"')
            return f'"{tok_esc}"'

        table_fixed = re.sub(pattern, fix_token, table_str)

        # 3) 安全解析
        table = ast.literal_eval(table_fixed)

        # 4) 基本结构校验：二维 list，且至少 2x2
        if not isinstance(table, list) or not all(isinstance(row, list) for row in table):
            return None
        if len(table) < 2 or len(table[0]) < 2:
            return None

        # 5) 类型规范化：第一行全转 str；第一列全转 str；其余尽量 float（缺失补 None）
        norm = []
        header = [str(x) for x in table[0]]
        norm.append(header)

        cols_n = len(header)

        for r in range(1, len(table)):
            row = table[r]
            if not isinstance(row, list) or len(row) == 0:
                return None

            row_label = str(row[0])
            new_row = [row_label]

            for j in range(1, cols_n):
                v = row[j] if j < len(row) else None
                if v is None:
                    new_row.append(None)
                    continue
                if isinstance(v, (int, float)):
                    new_row.append(float(v))
                elif isinstance(v, str):
                    vv = v.strip()
                    if _NUM.fullmatch(vv):
                        new_row.append(float(vv))
                    else:
                        new_row.append(v)
                else:
                    return None

            norm.append(new_row)

        return norm

    except Exception as e:
        print("Parse error:", e)
        return None



def evaluate_table_list(output, gt):
    
    pred_table = parse_table_list(output)
    
    # print("-----pred_table------")
    # print(pred_table)
    
    # 维度不一致也没关系 
    
    pred_cells = table_to_cells(pred_table)
    gold_cells = table_to_cells(gt)

    
    # print("-----pred_cells----")
    # print(pred_cells)
    # print("----gold_cells-----")
    # print(gold_cells)
    
    n = len(gold_cells)
    m = len(pred_cells)
    
    # print(f"n = {n}, m = {m}")
    
    if n == 0 and m == 0:
        return 1.0
    if n == 0 or m == 0:
        return 0.0

    # weight[i][j] = sim(gold_i, pred_j)
    weight = [[0.0 for _ in range(m)] for __ in range(n)]

    for i, (gr, gc, gv) in enumerate(gold_cells):
        gkey = f"{gr} | {gc}"
        for j, (pr, pc, pv) in enumerate(pred_cells):
            pkey = f"{pr} | {pc}"

            key_sim = esm_sim(gkey, pkey)

            ns = num_sim(pv, gv)
            weight[i][j] = key_sim * ns

    assignment = greedy_max(weight)

    matched_sum = 0.0
    for i in range(n):
        j = assignment[i]
        if j != -1 and weight[i][j] > 0:
            matched_sum += weight[i][j]

    precision = matched_sum / m
    recall = matched_sum / n
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall / (precision + recall))
    return f1
    
    


In [ ]:
ROOT_PATH = "und_gen"

SUBJECT = "chart_to_table"
JSON_FILE = "comp_to_gen.json"


def evaluate_dataset(subject_path, full_json_path):

    with open(full_json_path, "r", encoding="utf-8") as f:
        samples_data = json.load(f)

    stats = {
        "perceptual": {"score": 0, "count": 0},
        "reasoning": {"score": 0, "count": 0},
        "table to chart": {"score": 0, "count": 0},
        "chart to table": {"score": 0, "count": 0},
    }

    for sample in samples_data:
        print("-----sample-----")
        print(sample)

        output = call_llm_multiround(subject_path, sample)
        gt = sample["answer"]
        
        print(f"answer2 = {output}\n")

        score = evaluate_table_list(output, gt)
        
        print(f"score = {score}\n")

        question_type = sample["question_type"]

        if question_type in stats:
            stats[question_type]["count"] += 1
            stats[question_type]["score"] += score

    return stats


# ===== 路径 =====
subject_path = os.path.join(ROOT_PATH, SUBJECT)
full_json_path = os.path.join(subject_path, JSON_FILE)

if not os.path.exists(full_json_path):
    raise FileNotFoundError(f"{full_json_path} 不存在")

print(f"\n====== Running {SUBJECT} | {JSON_FILE} ======")

# ===== 运行 =====
stats = evaluate_dataset(subject_path, full_json_path)


# ===== 输出结果 =====
print("\n====== Result ======")

for k in stats:
    total_score = stats[k]["score"]
    total_count = stats[k]["count"]

    avg = total_score / total_count if total_count else 0

    print(f"{k}: Count={total_count}, Avg={avg:.4f}")